# UC Food Waste · 채널별 성능 비교

GPU 런타임에서 아래 코드 셀을 Shift+Enter로 실행하세요. 세 실험의 결과를 Drive에 저장하고 Test 지표를 비교합니다.


In [ ]:
"""Locate Colab data and start the RGB+Thermal trial."""

import csv
import os
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile


def locate(names, configured):
    if configured:
        candidates = [Path(configured)]
    else:
        result = subprocess.run(
            ["find", "-L", "/content", "-type", "f", "(",
             "-name", "image000010.npz", "-o", "-name", "data_processed.zip", ")"],
            capture_output=True, text=True,
        )
        candidates = [Path(line) for line in result.stdout.splitlines()]
    folders = {p.parent if p.suffix == ".npz" else p for p in candidates}
    npz = next((p for p in sorted(folders)
                if all((p / f"{name}.npz").is_file() for name in names)), None)
    return npz, [p for p in candidates if p.name == "data_processed.zip"]


def main():
    with Path("metadata/data_split.csv").open(encoding="utf-8-sig") as stream:
        names = sorted(row["filename"] for row in csv.DictReader(stream))
    configured = os.environ.get("UC_FOODWASTE_NPZ_DIR")
    if not configured and Path("/content/drive/MyDrive/images/image000010.npz").is_file():
        configured = "/content/drive/MyDrive/images"
    print("Colab 데이터 위치 확인 중...", flush=True)
    npz, archives = locate(names, configured)
    if npz is None:
        raise SystemExit(
            "현재 Colab에서 split CSV에 대응하는 NPZ 542개가 있는 폴더를 찾지 못했습니다. "
            "NPZ가 다른 계정의 '공유 문서함'에 있으면 Google Drive에서 "
            "해당 폴더의 바로가기를 내 드라이브에 추가한 뒤 다시 실행하세요."
        )
    masks = Path("data/data_processed/masks")
    if not all((masks / f"{n}.png").is_file() for n in names):
        if configured:
            _, archives = locate([], None)
        for path in sorted(archives):
            with ZipFile(path) as archive:
                entries = {
                    Path(name).stem: name for name in archive.namelist()
                    if Path(name).parent.name == "masks" and Path(name).suffix.lower() == ".png"
                }
                if not entries.keys() >= set(names):
                    continue
                masks.mkdir(parents=True, exist_ok=True)
                for name in names:
                    target = masks / f"{name}.png"
                    if not target.is_file():
                        target.write_bytes(archive.read(entries[name]))
                break
        else:
            raise SystemExit("마스크 542개를 포함한 data_processed.zip을 현재 Colab에서 찾지 못했습니다.")
    print(f"NPZ 폴더: {npz}", flush=True)
    print(f"마스크 폴더: {masks.resolve()}", flush=True)
    return npz.absolute(), masks.absolute()


from google.colab import drive
from IPython.display import display, Markdown
import torch

project = Path('/content/UC-FoodWaste')
if not torch.cuda.is_available():
    raise RuntimeError('런타임 → 런타임 유형 변경에서 GPU를 선택하세요.')

print('GPU:', torch.cuda.get_device_name(0), flush=True)
if not (project / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/jeehyung0922-lab/UC-FoodWaste.git', str(project)], check=True)
if not (project / 'archive/unet/scripts/train.py').is_file():
    raise RuntimeError('클론 폴더 안에 archive/unet/scripts/train.py가 없습니다.')
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

print('학습 패키지 확인 중...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'archive/unet/requirements.txt')], check=True)
from datetime import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt

MODES = ("rgb", "rgb_thermal", "all19")
EPOCHS = 50
BATCH_SIZE = 2
SEED = 42

previous_dir = Path.cwd()
try:
    os.chdir(project)
    npz, masks = main()
finally:
    os.chdir(previous_dir)

output = Path('/content/drive/MyDrive/UC-FoodWaste-runs') / datetime.now().strftime('comparison_%Y%m%d_%H%M%S_%f')
output.mkdir(parents=True, exist_ok=False)
print('Drive 저장 위치:', output, flush=True)
rows = []
for mode in MODES:
    print(f'\n=== {mode} ===', flush=True)
    result = output / mode
    result.mkdir()
    log_path = result / 'train.log'
    command = [
        sys.executable, '-u', 'archive/unet/scripts/train.py', '--mode', mode,
        '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
        '--seed', str(SEED), '--num-workers', '0', '--patience', '8',
        '--npz-dir', str(npz), '--mask-dir', str(masks),
        '--output-dir', str(output),
    ]
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        with subprocess.Popen(command, cwd=project, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, errors='replace') as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            status = process.wait()
    if status != 0:
        raise RuntimeError(f'{mode} 학습 종료 (코드 {status}). 위 학습 출력 또는 {log_path}를 확인하세요.')
    summary = json.loads((result / 'test_metrics.json').read_text())
    history = pd.read_csv(result / 'history.csv')
    best = history.loc[history['val_iou'].idxmax()]
    rows.append({
        'mode': mode, 'best_epoch': int(best['epoch']),
        'val_iou': best['val_iou'], **summary['overall'],
    })
    pd.DataFrame(rows).to_csv(output / 'comparison.csv', index=False)

table = pd.DataFrame(rows).set_index('mode')
display(Markdown('### 채널별 성능 비교'))
display(table.round(4))
ax = table[['iou', 'dice', 'recall', 'precision']].plot.bar(
    rot=0, ylim=(0, 1), figsize=(8, 4), ylabel='Test score'
)
ax.figure.tight_layout()
ax.figure.savefig(output / 'comparison.png', dpi=150)
plt.show()
print('Drive 저장 완료:', output)
